# Plotting the Cell Voltage Curve Against OCV (Graphite)

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# PARSER
# ============================================================

pattern = re.compile(
    r"timestep:\s*\d+\s*\[.*?\],\s*VCell\s*=\s*([-+eE0-9.]+).*?"
    r"Cp_min\s*=\s*([-+eE0-9.]+).*?"
    r"Cp_max\s*=\s*([-+eE0-9.]+).*?"
    r"XfrC_avg\s*=\s*([-+eE0-9.]+)",
    re.DOTALL
)

def load_data(file_path):
    with open(file_path, "r") as f:
        text = f.read()

    matches = pattern.findall(text)

    data = []
    for vcell, cp_min, cp_max, xfrc in matches:
        data.append((
            float(xfrc),
            float(vcell),
            float(cp_min),
            float(cp_max)
        ))

    df = pd.DataFrame(
        data,
        columns=["XfrC_avg", "VCell", "Cp_min", "Cp_max"]
    )

    print(f"{file_path}")
    print(f"Points found: {len(df)}")
    print(df.head())

    return df

# ============================================================
# LOAD FILES
# ============================================================

df1 = load_data(
    "../MSU_HPCC/slurm_besfem-GC-TEST-e-16-12hrs-12893611.SLURMout"
)


# ============================================================
# STEP 2: LOAD Graphite OCV TABLE
# ============================================================

ticks = np.loadtxt("../inputs/materials/C_Li_X_101.txt")
ocv = np.loadtxt("../inputs/materials/C_Li_O3_101.txt")

# ============================================================
# STEP 3: INTERPOLATE OCV AT SIMULATION XfrC VALUES
# ============================================================

df1["ocv"] = np.interp(df1["XfrC_avg"], ticks, ocv)

# ============================================================
# STEP 4: PLOT
# ============================================================

plt.figure(figsize=(10,6))

# Simulation voltage
plt.plot(
    df1["XfrC_avg"],
    df1["VCell"],
    linewidth=2,
    label="Simulation VCell"
)

# OCV curve
plt.plot(
    ticks,
    ocv,
    '--',
    linewidth=1,
    label="Graphite OCV"
)

plt.xlabel("XfrC")
plt.ylabel("Voltage (V)")
plt.title("VCell vs XfrC with OCV")
plt.grid(True)
plt.legend()
plt.tight_layout()

plt.show()

# Plotting the Cell Voltage Curve Against OCV (NMC)

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# PARSER
# ============================================================

pattern = re.compile(
    r"timestep:\s*\d+\s*\[.*?\],\s*VCell\s*=\s*([-+eE0-9.]+).*?"
    r"Cp_min\s*=\s*([-+eE0-9.]+).*?"
    r"Cp_max\s*=\s*([-+eE0-9.]+).*?"
    r"XfrC_avg\s*=\s*([-+eE0-9.]+)",
    re.DOTALL
)

def load_data(file_path):
    with open(file_path, "r") as f:
        text = f.read()

    matches = pattern.findall(text)

    data = []
    for vcell, cp_min, cp_max, xfrc in matches:
        data.append((
            float(xfrc),
            float(vcell),
            float(cp_min),
            float(cp_max)
        ))

    df = pd.DataFrame(
        data,
        columns=["XfrC_avg", "VCell", "Cp_min", "Cp_max"]
    )

    print(f"{file_path}")
    print(f"Points found: {len(df)}")
    print(df.head())

    return df

# ============================================================
# LOAD FILES
# ============================================================

df1 = load_data(
    "../MSU_HPCC/slurm_besfem-GC-TEST-e-16-12hrs-12893611.SLURMout"
)

# ============================================================
# STEP 2: DEFINE OCV FUNCTION
# ============================================================

def ocv_function(c):
    return (
        1.095 * c * c
        - 8.234e-7 * np.exp(14.31 * c)
        + 4.692 * np.exp(-0.5389 * c)
    )

# Generate smooth OCV curve
c_vals = np.linspace(0.01, 1.0, 500)
ocv_vals = ocv_function(c_vals)

# Interpolated OCV at simulation points
df1["ocv"] = ocv_function(df1["XfrC_avg"])

# ============================================================
# PLOT 1: VOLTAGE VS AVERAGE LITHIATION
# ============================================================

plt.figure(figsize=(8, 6))

plt.plot(
    df1["XfrC_avg"],
    df1["VCell"],
    linewidth=2,
    label="Simulation"
)

plt.plot(
    c_vals,
    ocv_vals,
    "--",
    linewidth=1,
    label="NMC OCV"
)

plt.xlabel("XfrC_avg")
plt.ylabel("Voltage (V)")
plt.title("Voltage vs Average Lithiation")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


# Plotting the Cell Voltage Curve Against OCV (LFP)

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# PARSER
# ============================================================

pattern = re.compile(
    r"timestep:\s*\d+\s*\[.*?\],\s*VCell\s*=\s*([-+eE0-9.]+).*?"
    r"Cp_min\s*=\s*([-+eE0-9.]+).*?"
    r"Cp_max\s*=\s*([-+eE0-9.]+).*?"
    r"XfrC_avg\s*=\s*([-+eE0-9.]+)",
    re.DOTALL
)

def load_data(file_path):
    with open(file_path, "r") as f:
        text = f.read()

    matches = pattern.findall(text)

    data = []
    for vcell, cp_min, cp_max, xfrc in matches:
        data.append((
            float(xfrc),
            float(vcell),
            float(cp_min),
            float(cp_max)
        ))

    df = pd.DataFrame(
        data,
        columns=["XfrC_avg", "VCell", "Cp_min", "Cp_max"]
    )

    print(f"{file_path}")
    print(f"Points found: {len(df)}")
    print(df.head())

    return df

# ============================================================
# LOAD FILES
# ============================================================

df1 = load_data(
    "../MSU_HPCC/slurm_besfem-GC-TEST-e-16-12hrs-12893611.SLURMout"
)

# ============================================================
# LOAD AND CALCULATE LFP OCV
# ============================================================

ticks = np.loadtxt("../inputs/materials/LFP_Chm_Pot_Ticks.txt")
chmPot = np.loadtxt("../inputs/materials/LFP_Chm_Pot.txt")

# Matches MaterialProperties.cpp:
# OCV = -chemical_potential + 3.4
ocv = -chmPot + 3.4

# ============================================================
# PLOT 1: VOLTAGE VS AVERAGE LITHIATION
# ============================================================

plt.figure(figsize=(8, 6))

plt.plot(
    df1["XfrC_avg"],
    df1["VCell"],
    linewidth=2,
    label="Simulation"
)

plt.plot(
    ticks,
    ocv,
    "--",
    linewidth=1,
    label="LFP OCV"
)

plt.xlabel("XfrC_avg")
plt.ylabel("Voltage (V)")
plt.title("Voltage vs Average Lithiation")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
